# 23-22 · Тесты дубликатов и пустых файлов

Практика к разделу [«Проверяем поиск дубликатов»](../../site/chapters/glava-23/23-26-testy-dublikatov.html). Повторяет `projects/python/safesort/tests/test_duplicates.py`, но полностью в памяти — без обращения к диску.

## Цель

Написать и запустить тесты для группировки дубликатов: типичный случай, файлы без пары и — отдельно — пустые файлы, которые тоже считаются дубликатами друг друга.

## Рабочий пример

In [1]:
import hashlib
from collections import defaultdict
from dataclasses import dataclass


@dataclass(frozen=True)
class FileInfo:
    name: str
    size: int
    content: bytes


def find_duplicates(files):
    by_size = defaultdict(list)
    for file in files:
        by_size[file.size].append(file)

    groups = []
    for size, candidates in by_size.items():
        if len(candidates) < 2:
            continue
        by_digest = defaultdict(list)
        for candidate in candidates:
            digest = hashlib.sha256(candidate.content).hexdigest()
            by_digest[digest].append(candidate)
        for digest, matched in by_digest.items():
            if len(matched) >= 2:
                groups.append({"size": size, "digest": digest, "files": tuple(matched)})
    return groups


def test_identical_content_files_are_grouped():
    files = [
        FileInfo("notes.txt", 12, b"tot zhe text"),
        FileInfo("copy_of_notes.txt", 12, b"tot zhe text"),
    ]
    groups = find_duplicates(files)
    assert len(groups) == 1
    assert len(groups[0]["files"]) == 2


def test_different_content_same_size_not_grouped():
    files = [
        FileInfo("a.txt", 4, b"AAAA"),
        FileInfo("b.txt", 4, b"BBBB"),
    ]
    groups = find_duplicates(files)
    assert groups == []


def test_empty_files_are_duplicates_of_each_other():
    files = [
        FileInfo("a.txt", 0, b""),
        FileInfo("b.txt", 0, b""),
    ]
    groups = find_duplicates(files)
    assert len(groups) == 1
    assert groups[0]["size"] == 0
    assert groups[0]["digest"] == hashlib.sha256(b"").hexdigest()


for test_func in (
    test_identical_content_files_are_grouped,
    test_different_content_same_size_not_grouped,
    test_empty_files_are_duplicates_of_each_other,
):
    test_func()
    print(f"OK: {test_func.__name__}")

OK: test_identical_content_files_are_grouped
OK: test_different_content_same_size_not_grouped
OK: test_empty_files_are_duplicates_of_each_other


## Проверка результата

In [2]:
groups_dlya_proverki = find_duplicates([
    FileInfo("x1.bin", 3, b"XXX"),
    FileInfo("x2.bin", 3, b"XXX"),
    FileInfo("x3.bin", 3, b"XXX"),
])

assert len(groups_dlya_proverki) == 1
assert len(groups_dlya_proverki[0]["files"]) == 3
print("Верно: три файла с одинаковым содержимым образовали одну группу из трёх, "
      "а не полтора дубликата.")

Верно: три файла с одинаковым содержимым образовали одну группу из трёх, а не полтора дубликата.


## Задание ★★ Самостоятельная задача

In [3]:
def test_three_size_groups_only_two_have_duplicates():
    files = [
        FileInfo("a1.txt", 5, b"AAAAA"),
        FileInfo("a2.txt", 5, b"AAAAA"),
        FileInfo("b1.txt", 7, b"BBBBBBB"),
        FileInfo("c1.txt", 9, b"CCCCCCCCC"),  # уникальный размер — не дубликат
    ]
    groups = find_duplicates(files)
    assert len(groups) == 1
    assert {f.name for f in groups[0]["files"]} == {"a1.txt", "a2.txt"}


test_three_size_groups_only_two_have_duplicates()
print("OK: test_three_size_groups_only_two_have_duplicates")

OK: test_three_size_groups_only_two_have_duplicates
